# Python — Advanced Reference for Data Science
> **Level:** Advanced | **Goal:** Master Python patterns essential for production data science code

## Table of Contents
1. [Pythonic Idioms](#idioms)
2. [Decorators & Closures](#decorators)
3. [Generators & Itertools](#generators)
4. [Context Managers](#context)
5. [Type Hints & Dataclasses](#types)
6. [NumPy — Vectorization & Broadcasting](#numpy)
7. [Pandas — Advanced Operations](#pandas)
8. [Performance: Profiling & Optimization](#performance)
9. [Concurrency for Data Pipelines](#concurrency)

In [ ]:
import numpy as np
import pandas as pd
import time, functools, itertools, collections
from typing import Any, Callable, Generator, Iterator, TypeVar
from dataclasses import dataclass, field
from contextlib import contextmanager

print("NumPy", np.__version__, "| Pandas", pd.__version__)

---
## 1 · Pythonic Idioms <a id='idioms'></a>

In [ ]:
# ── Comprehensions ─────────────────────────────────────────────
# List comprehension with condition
evens = [x**2 for x in range(20) if x % 2 == 0]

# Dict comprehension
word_len = {word: len(word) for word in ['alpha','beta','gamma','delta']}

# Set comprehension (deduplication)
unique_mods = {x % 5 for x in range(30)}

# Generator expression (lazy — no memory allocation)
total = sum(x**2 for x in range(10**7))  # doesn't build a list

# Nested comprehension
matrix = [[i * j for j in range(1, 6)] for i in range(1, 6)]
print("Matrix row 2:", matrix[1])

In [ ]:
# ── Unpacking ──────────────────────────────────────────────────
first, *middle, last = [1, 2, 3, 4, 5]
print(f"first={first}, middle={middle}, last={last}")

# Swap without temp variable
a, b = 10, 20
a, b = b, a

# Unpack nested
(x, y), z = (1, 2), 3

# Dictionary merge (Python 3.9+)
defaults = {"lr": 0.001, "epochs": 10, "batch_size": 32}
overrides = {"epochs": 50, "dropout": 0.3}
config = defaults | overrides
print(config)

In [ ]:
# ── collections module ─────────────────────────────────────────
from collections import Counter, defaultdict, deque, namedtuple, OrderedDict

# Counter — frequency map
tokens = ['cat','dog','cat','fish','dog','cat']
c = Counter(tokens)
print("Most common:", c.most_common(2))

# defaultdict — avoid KeyError
groups = defaultdict(list)
for name, dept in [('Alice','Eng'),('Bob','HR'),('Carol','Eng')]:
    groups[dept].append(name)
print(dict(groups))

# namedtuple — lightweight immutable record
Point = namedtuple('Point', ['x', 'y', 'z'])
p = Point(1.0, 2.5, -3.0)
print(p.x, p._asdict())

---
## 2 · Decorators & Closures <a id='decorators'></a>

In [ ]:
# ── Basic decorator ────────────────────────────────────────────
def timer(func: Callable) -> Callable:
    """Measure execution time of any function."""
    @functools.wraps(func)  # preserve __name__, __doc__
    def wrapper(*args, **kwargs):
        t0 = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - t0
        print(f"[{func.__name__}] {elapsed:.4f}s")
        return result
    return wrapper

@timer
def expensive_computation(n: int) -> float:
    return sum(i**0.5 for i in range(n))

expensive_computation(1_000_000)

In [ ]:
# ── Decorator with arguments ───────────────────────────────────
def retry(max_attempts: int = 3, exceptions: tuple = (Exception,)):
    """Retry decorator for transient failures (e.g., network calls)."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    if attempt == max_attempts:
                        raise
                    print(f"Attempt {attempt} failed: {e}. Retrying...")
        return wrapper
    return decorator

@retry(max_attempts=3, exceptions=(ValueError,))
def flaky_function(x):
    if x < 0.7:
        raise ValueError("Random failure")
    return "success"

import random; random.seed(42)
try:
    result = flaky_function(random.random())
    print(result)
except ValueError:
    print("All retries exhausted")

In [ ]:
# ── Caching with functools.lru_cache / cache ───────────────────
@functools.lru_cache(maxsize=128)
def fibonacci(n: int) -> int:
    if n < 2: return n
    return fibonacci(n - 1) + fibonacci(n - 2)

print([fibonacci(i) for i in range(15)])
print(fibonacci.cache_info())

---
## 3 · Generators & Itertools <a id='generators'></a>

In [ ]:
# ── Generator function: memory-efficient data pipeline ─────────
def read_chunks(data: list, chunk_size: int) -> Generator:
    """Yield successive chunks — useful for batch processing large files."""
    for i in range(0, len(data), chunk_size):
        yield data[i : i + chunk_size]

def preprocess(chunk: list) -> list:
    return [x * 2 for x in chunk if x % 3 != 0]

data = list(range(100))
pipeline = (preprocess(chunk) for chunk in read_chunks(data, 10))
results = list(pipeline)
print(f"{len(results)} chunks processed, first chunk: {results[0]}")

In [ ]:
# ── itertools essentials ───────────────────────────────────────
from itertools import (
    chain, islice, groupby, combinations, permutations,
    product, accumulate, takewhile, dropwhile, zip_longest
)

# chain — flatten nested iterables
flat = list(chain([1,2], [3,4], [5,6]))
print("chain:", flat)

# islice — lazy slicing (no list creation)
first_five = list(islice((x**2 for x in range(100)), 5))
print("islice:", first_five)

# accumulate — running totals / prefix sums
import operator
prefix_sum = list(accumulate([1,2,3,4,5]))
prefix_prod = list(accumulate([1,2,3,4,5], operator.mul))
print("prefix_sum:", prefix_sum)
print("prefix_prod:", prefix_prod)

# groupby — group consecutive elements (sort first!)
rows = sorted([('Eng','Alice'),('HR','Bob'),('Eng','Carol'),('HR','Eve')], key=lambda x: x[0])
for dept, members in groupby(rows, key=lambda x: x[0]):
    print(f"{dept}: {[m[1] for m in members]}")

---
## 4 · Context Managers <a id='context'></a>

In [ ]:
# ── @contextmanager — simplest way ─────────────────────────────
@contextmanager
def timer_ctx(label: str):
    """Context manager that times a code block."""
    t0 = time.perf_counter()
    try:
        yield
    finally:
        print(f"[{label}] {time.perf_counter() - t0:.4f}s")

with timer_ctx("array creation"):
    arr = np.random.randn(10_000_000)

# ── Class-based context manager ────────────────────────────────
class ManagedResource:
    def __init__(self, name: str):
        self.name = name

    def __enter__(self):
        print(f"Acquiring {self.name}")
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        print(f"Releasing {self.name}")
        return False  # don't suppress exceptions

with ManagedResource("DB connection") as r:
    print(f"Using {r.name}")

---
## 5 · Type Hints & Dataclasses <a id='types'></a>

In [ ]:
from __future__ import annotations
from dataclasses import dataclass, field
from typing import ClassVar, Optional

@dataclass(order=True, frozen=False)
class ModelConfig:
    """Typed configuration object for ML models."""
    # ClassVar is not an instance field
    VALID_OBJECTIVES: ClassVar[set] = {'regression', 'classification', 'ranking'}

    model_name: str
    learning_rate: float = 0.001
    n_estimators: int = 100
    max_depth: Optional[int] = None
    features: list[str] = field(default_factory=list)  # mutable default
    objective: str = 'regression'

    def __post_init__(self):
        if self.objective not in self.VALID_OBJECTIVES:
            raise ValueError(f"objective must be one of {self.VALID_OBJECTIVES}")
        if self.learning_rate <= 0:
            raise ValueError("learning_rate must be positive")

    def to_dict(self) -> dict:
        from dataclasses import asdict
        return asdict(self)

cfg = ModelConfig('xgboost', learning_rate=0.05, n_estimators=500, features=['age','income'])
print(cfg)
print(cfg.to_dict())

---
## 6 · NumPy — Vectorization & Broadcasting <a id='numpy'></a>

In [ ]:
rng = np.random.default_rng(42)

# ── Broadcasting ───────────────────────────────────────────────
# Rule: shapes are compatible if each dimension is equal OR one of them is 1
A = rng.integers(0, 10, (4, 3))
row_means = A.mean(axis=1, keepdims=True)  # shape (4,1)
centered = A - row_means                    # broadcast (4,3) - (4,1) → (4,3)
print("A:\n", A)
print("Centered:\n", centered)

In [ ]:
# ── Vectorized operations vs Python loops ─────────────────────
n = 10_000_000
arr = rng.standard_normal(n)

with timer_ctx("Python loop"):
    total_py = sum(x**2 for x in arr)

with timer_ctx("NumPy vectorized"):
    total_np = (arr**2).sum()

print(f"Results match: {abs(total_py - total_np) < 1e-3}")

In [ ]:
# ── Fancy indexing & boolean masks ────────────────────────────
data = rng.standard_normal((1000, 5))

# Boolean mask
mask = data[:, 0] > 1.5       # rows where column 0 > 1.5
filtered = data[mask]
print(f"Rows with col[0] > 1.5: {len(filtered)}")

# np.where — vectorized if/else
labels = np.where(data[:, 0] > 0, 1, 0)

# np.select — vectorized multi-condition
conditions = [data[:, 0] > 1, (data[:, 0] > -1) & (data[:, 0] <= 1), data[:, 0] <= -1]
choices    = ['high', 'medium', 'low']
category   = np.select(conditions, choices)
print(collections.Counter(category))

In [ ]:
# ── Linear algebra essentials ──────────────────────────────────
X = rng.standard_normal((100, 5))
y = rng.standard_normal(100)

# OLS solution: β = (XᵀX)⁻¹ Xᵀy
beta_ols = np.linalg.lstsq(X, y, rcond=None)[0]

# SVD
U, s, Vt = np.linalg.svd(X, full_matrices=False)
print("Singular values:", s.round(2))

# Einsum — expressive tensor operations
# Batch matrix multiply: (B, M, K) @ (B, K, N) -> (B, M, N)
A = rng.standard_normal((10, 4, 3))
B = rng.standard_normal((10, 3, 5))
C = np.einsum('bmk,bkn->bmn', A, B)
print("Batch matmul shape:", C.shape)

---
## 7 · Pandas — Advanced Operations <a id='pandas'></a>

In [ ]:
# ── Sample dataset ────────────────────────────────────────────
rng2 = np.random.default_rng(0)
n = 5000
df = pd.DataFrame({
    'user_id':  rng2.integers(1, 200, n),
    'product':  rng2.choice(['A','B','C','D'], n),
    'category': rng2.choice(['Electronics','Clothing','Food'], n),
    'revenue':  rng2.exponential(100, n).round(2),
    'date':     pd.date_range('2023-01-01', periods=n, freq='1h'),
    'region':   rng2.choice(['North','South','East','West'], n),
})
df.head()

In [ ]:
# ── groupby: transform vs aggregate ──────────────────────────
# agg: reduces each group to one row
agg = df.groupby('category')['revenue'].agg(['mean','median','std','count'])
print(agg.round(2))

# transform: returns same-size series (broadcast back to original df)
df['cat_mean_rev'] = df.groupby('category')['revenue'].transform('mean')
df['rev_vs_mean']  = df['revenue'] / df['cat_mean_rev']

# Named aggregation (pandas 0.25+)
summary = df.groupby(['category','region']).agg(
    total_rev   = ('revenue', 'sum'),
    avg_rev     = ('revenue', 'mean'),
    n_orders    = ('revenue', 'count'),
    unique_users= ('user_id', 'nunique')
).round(2)
summary.head(8)

In [ ]:
# ── pivot_table vs crosstab ─────────────────────────────────
pivot = df.pivot_table(
    values='revenue',
    index='category',
    columns='region',
    aggfunc='mean',
    margins=True
).round(2)
print(pivot)

# crosstab — frequency tables
ct = pd.crosstab(df['category'], df['product'], normalize='index').round(3)
print("\nProduct share per category:")
print(ct)

In [ ]:
# ── Time series resampling & rolling ─────────────────────────
ts = df.set_index('date')['revenue']

# Resample to daily totals
daily = ts.resample('D').sum()

# Rolling statistics
rolling = daily.rolling(window=7, min_periods=1)
daily_stats = pd.DataFrame({
    'revenue':     daily,
    'roll_mean_7': rolling.mean(),
    'roll_std_7':  rolling.std(),
    'ewm_14':      daily.ewm(span=14).mean()
})
print(daily_stats.head(14).round(2))

In [ ]:
# ── apply with multiple return values ─────────────────────────
def revenue_stats(group):
    return pd.Series({
        'p25': group.quantile(0.25),
        'p50': group.median(),
        'p75': group.quantile(0.75),
        'iqr': group.quantile(0.75) - group.quantile(0.25),
        'skewness': group.skew()
    })

df.groupby('category')['revenue'].apply(revenue_stats).round(2)

In [ ]:
# ── Memory optimization ────────────────────────────────────────
print("Before optimization:")
print(df.dtypes)
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e6:.2f} MB")

# Convert object → category for low-cardinality strings
for col in ['product','category','region']:
    df[col] = df[col].astype('category')

# Downcast numerics
df['user_id'] = pd.to_numeric(df['user_id'], downcast='integer')

print("\nAfter optimization:")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e6:.2f} MB")

---
## 8 · Performance: Profiling & Optimization <a id='performance'></a>

In [ ]:
# ── %timeit for micro-benchmarks ─────────────────────────────
arr = np.random.randn(100_000)

%timeit np.sum(arr)
%timeit sum(arr)          # pure Python — much slower

In [ ]:
# ── cProfile + pstats ─────────────────────────────────────────
import cProfile, pstats, io

def workload():
    data = [x**0.5 for x in range(100_000)]
    return sorted(data, reverse=True)[:10]

pr = cProfile.Profile()
pr.enable()
workload()
pr.disable()

buf = io.StringIO()
pstats.Stats(pr, stream=buf).sort_stats('cumulative').print_stats(10)
print(buf.getvalue())

---
## 9 · Concurrency for Data Pipelines <a id='concurrency'></a>

| Use case | Best tool |
|---|---|
| I/O bound (API calls, DB queries) | `asyncio` / `ThreadPoolExecutor` |
| CPU bound (feature engineering) | `ProcessPoolExecutor` / `multiprocessing` |
| DataFrames in parallel | `joblib` / `pandarallel` |

In [ ]:
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed

# ── ThreadPoolExecutor — I/O-bound parallel work ──────────────
def fetch_data(user_id: int) -> dict:
    """Simulate an API call (I/O bound)."""
    time.sleep(0.01)  # simulated latency
    return {'user_id': user_id, 'score': user_id * 1.5}

user_ids = list(range(1, 51))

with timer_ctx("sequential"):
    results_seq = [fetch_data(uid) for uid in user_ids]

with timer_ctx("ThreadPool (10 workers)"):
    with ThreadPoolExecutor(max_workers=10) as ex:
        futures = {ex.submit(fetch_data, uid): uid for uid in user_ids}
        results_par = [f.result() for f in as_completed(futures)]

print(f"Both approaches returned {len(results_seq)} results")

In [ ]:
# ── asyncio — async/await for I/O pipelines ───────────────────
import asyncio

async def async_fetch(session_id: int) -> dict:
    await asyncio.sleep(0.01)  # simulated async I/O
    return {'session': session_id, 'value': session_id**2}

async def run_pipeline(ids):
    tasks = [async_fetch(i) for i in ids]
    return await asyncio.gather(*tasks)

# In a Jupyter notebook, use await directly
results = await run_pipeline(range(20))
print(f"Fetched {len(results)} results. First: {results[0]}")

---
## How to Access Databases Using Python <a id='db-access'></a>

Python connects to databases through the **DB-API 2.0** standard (PEP 249).  
Every database driver follows the same interface — swap the driver, keep the code.

### The standard workflow

```
1. import driver
2. connection  = driver.connect(...)
3. cursor      = connection.cursor()
4. cursor.execute(sql)
5. rows        = cursor.fetchall()
6. connection.close()
```

### Key objects

| Object | Purpose | Key methods |
|---|---|---|
| **Connection** | Session with the database | `cursor()`, `commit()`, `rollback()`, `close()` |
| **Cursor** | Executes queries & holds results | `execute()`, `executemany()`, `fetchone()`, `fetchmany(n)`, `fetchall()` |

### Fetch methods

| Method | Returns |
|---|---|
| `fetchone()` | Next single row as a tuple (or `None`) |
| `fetchmany(n)` | Next `n` rows as a list of tuples |
| `fetchall()` | All remaining rows as a list of tuples |

### Popular Python database drivers

| Database | Driver | Install |
|---|---|---|
| SQLite (built-in) | `sqlite3` | — (stdlib) |
| PostgreSQL | `psycopg2` | `pip install psycopg2-binary` |
| MySQL / MariaDB | `pymysql` | `pip install pymysql` |
| SQL Server | `pyodbc` | `pip install pyodbc` |
| Any (SQLAlchemy ORM) | `sqlalchemy` | `pip install sqlalchemy` |

### Parameterized queries — always use these to prevent SQL injection

```python
# ✅ Safe — use ? placeholders (sqlite3) or %s (psycopg2/pymysql)
cursor.execute("SELECT * FROM users WHERE name = ?", (name,))

# ❌ Dangerous — never format strings directly into SQL
cursor.execute(f"SELECT * FROM users WHERE name = '{name}'")
```

---
## Benefits of Python for Database Programming <a id='python-db-benefits'></a>

| Benefit | Detail |
|---|---|
| **Rich ecosystem** | NumPy, pandas, matplotlib, SciPy — analyse and visualise query results without leaving Python |
| **Ease of use** | Clean, readable syntax; fewer lines of code than Java or C++ to achieve the same database task |
| **Portable** | Runs on Windows, macOS, Linux; scripts move between environments without modification |
| **Supports relational databases** | Works with SQLite, PostgreSQL, MySQL, SQL Server, Oracle, and more via standard drivers |
| **Python DB-API (PEP 249)** | A single, consistent interface for all databases — learn once, apply to any driver |

### What is DB-API?
The **Python Database API Specification (PEP 249)** defines a standard that all database drivers must follow.  
This means you can switch from SQLite to PostgreSQL by changing only the `import` and `connect()` call — the rest of your code stays the same.

```
import sqlite3           →   import psycopg2
sqlite3.connect(...)     →   psycopg2.connect(host=..., dbname=..., user=..., password=...)
cursor.execute(sql)      →   same
cursor.fetchall()        →   same
```

### The data pipeline Python enables

```
SQL Database
    ↓  sqlite3 / psycopg2 / pymysql
Python (DB-API cursor)
    ↓  pd.read_sql_query()
pandas DataFrame
    ↓  NumPy / SciPy
Analysis & Statistics
    ↓  matplotlib / seaborn / plotly
Visualisation & Reports
```

In [ ]:
# ── How to Access Databases Using Python ───────────────────────
import sqlite3
import pandas as pd

# ══════════════════════════════════════════════════════════════
# 1. Connect & create a cursor
# ══════════════════════════════════════════════════════════════
conn = sqlite3.connect(":memory:")   # use a file path for persistent DB
cur  = conn.cursor()
print("✅ Connected to SQLite", sqlite3.sqlite_version)

# ══════════════════════════════════════════════════════════════
# 2. Create table & insert rows
# ══════════════════════════════════════════════════════════════
cur.execute("""
    CREATE TABLE employees (
        id    INTEGER PRIMARY KEY,
        name  TEXT,
        dept  TEXT,
        salary REAL
    )
""")

rows = [
    (1, 'Alice',  'Analytics',   82000),
    (2, 'Bob',    'Engineering', 95000),
    (3, 'Carol',  'Analytics',   78000),
    (4, 'David',  'Marketing',   67000),
    (5, 'Eva',    'Engineering', 102000),
]
cur.executemany("INSERT INTO employees VALUES (?, ?, ?, ?)", rows)
conn.commit()
print("✅ Table created and 5 rows inserted")

# ══════════════════════════════════════════════════════════════
# 3. fetchone / fetchmany / fetchall
# ══════════════════════════════════════════════════════════════
cur.execute("SELECT * FROM employees")
print("\n── fetchone() ── (returns 1 row as tuple)")
print(cur.fetchone())

print("\n── fetchmany(2) ── (returns next 2 rows)")
print(cur.fetchmany(2))

print("\n── fetchall() ── (returns all remaining rows)")
print(cur.fetchall())

# ══════════════════════════════════════════════════════════════
# 4. Parameterized query — safe, prevents SQL injection
# ══════════════════════════════════════════════════════════════
dept_filter = "Analytics"
cur.execute("SELECT name, salary FROM employees WHERE dept = ?", (dept_filter,))
print(f"\n── Parameterized query: dept = '{dept_filter}' ──")
for row in cur.fetchall():
    print(row)

# ══════════════════════════════════════════════════════════════
# 5. Load results directly into a pandas DataFrame
# ══════════════════════════════════════════════════════════════
print("\n── pd.read_sql_query() → DataFrame ──")
df = pd.read_sql_query("SELECT * FROM employees ORDER BY salary DESC", conn)
display(df)

# ══════════════════════════════════════════════════════════════
# 6. Context manager — auto-commits and closes
# ══════════════════════════════════════════════════════════════
print("── Context manager (with statement) ──")
with sqlite3.connect(":memory:") as con2:
    con2.execute("CREATE TABLE test (x INTEGER)")
    con2.execute("INSERT INTO test VALUES (42)")
    val = con2.execute("SELECT x FROM test").fetchone()
    print("Value from context manager connection:", val)
# connection is automatically closed here

conn.close()
print("\n✅ Original connection closed")

# ══════════════════════════════════════════════════════════════
# 7. Connecting to other databases (reference — no install needed)
# ══════════════════════════════════════════════════════════════
print("""
── Other database drivers (syntax reference) ──

# PostgreSQL (psycopg2)
import psycopg2
conn = psycopg2.connect(host='localhost', dbname='mydb', user='user', password='pwd')

# MySQL (pymysql)
import pymysql
conn = pymysql.connect(host='localhost', db='mydb', user='user', password='pwd')

# SQL Server (pyodbc)
import pyodbc
conn = pyodbc.connect('DRIVER={SQL Server};SERVER=server;DATABASE=db;UID=user;PWD=pwd')

# All follow the same DB-API pattern: conn → cursor → execute → fetch → close
""")

---
## Introduction to Notebooks <a id='intro-notebooks'></a>

> Notebooks allow creating and sharing documents containing **live code, equations, visualisations, and explanatory text**.

### What a notebook is

A notebook is a document made up of **cells** that can hold:

| Cell type | Content |
|---|---|
| **Code** | Executable code (Python, R, SQL…) + output shown inline |
| **Markdown** | Formatted text, headings, bullet points, links |
| **Math** | LaTeX equations rendered inline: $E = mc^2$ |
| **Output** | Tables, charts, images — embedded directly below the code |

### Notebook environments

| Environment | Language(s) | Notes |
|---|---|---|
| **Jupyter Notebook / Lab** | Python, R, Julia, SQL… | Open-source, runs in browser, most widely used |
| **VS Code Notebooks** | Python + extensions | Native in VS Code, same `.ipynb` format |
| **MATLAB Live Editor** | MATLAB | Proprietary, similar concept |
| **Google Colab** | Python | Cloud-hosted Jupyter, free GPU access |
| **Databricks** | Python, SQL, Scala | Cloud, big-data focused |

### Why notebooks matter for data science

- **Reproducibility** — code, results, and explanation live together in one file
- **Exploration** — run one cell at a time; inspect intermediate results
- **Communication** — share `.ipynb` files; export to PDF / HTML
- **Education** — ideal for course notes (exactly what this knowledge base is!)

### Jupyter notebook file format
Notebooks are saved as `.ipynb` — a **JSON** file containing cells, outputs, and metadata.

```json
{
  "cells": [
    { "cell_type": "markdown", "source": ["# Hello"] },
    { "cell_type": "code",     "source": ["print('Hello')"], "outputs": [...] }
  ],
  "metadata": { "kernelspec": { "name": "python3" } }
}
```

---
## DB-API — Writing Code <a id='dbapi-writing-code'></a>

The DB-API defines **two core objects** you work with every time:

### Connection object
Created by calling the driver's `connect()` function.

```python
connection = sqlite3.connect("mydb.db")   # file-based
connection = sqlite3.connect(":memory:")  # in-memory (no file)
```

| Method | What it does |
|---|---|
| `connection.cursor()` | Returns a new Cursor object |
| `connection.commit()` | Saves pending changes to the DB |
| `connection.rollback()` | Undoes changes since last commit |
| `connection.close()` | Closes the connection |

### Cursor object
The cursor is the **workhorse** — it executes SQL and holds results.

```python
cursor = connection.cursor()
```

| Method | What it does |
|---|---|
| `cursor.execute(sql)` | Run one SQL statement |
| `cursor.execute(sql, params)` | Run with parameterized values |
| `cursor.executemany(sql, seq)` | Run same SQL for a list of param sets |
| `cursor.fetchone()` | Return next row as a tuple |
| `cursor.fetchmany(n)` | Return next `n` rows |
| `cursor.fetchall()` | Return all remaining rows |
| `cursor.description` | Column names & types of last query |
| `cursor.rowcount` | Rows affected by last INSERT/UPDATE/DELETE |

### Complete code pattern

```python
import sqlite3

# 1 — Connect
conn = sqlite3.connect(":memory:")

# 2 — Create cursor
cur = conn.cursor()

# 3 — Execute DDL
cur.execute("""
    CREATE TABLE books (id INTEGER PRIMARY KEY, title TEXT, year INTEGER)
""")

# 4 — Insert with parameters (safe — no SQL injection)
cur.execute("INSERT INTO books VALUES (?, ?, ?)", (1, "Dune", 1965))
conn.commit()

# 5 — Query
cur.execute("SELECT * FROM books WHERE year > ?", (1960,))
rows = cur.fetchall()
for row in rows:
    print(row)

# 6 — Column names from cursor.description
col_names = [desc[0] for desc in cur.description]
print("Columns:", col_names)

# 7 — Close
conn.close()
```

### Two ways to manage the connection

```python
# Option A — Manual (always close!)
conn = sqlite3.connect("mydb.db")
# ... work ...
conn.close()

# Option B — Context manager (recommended)
with sqlite3.connect("mydb.db") as conn:
    conn.execute("INSERT INTO books VALUES (2, 'Foundation', 1951)")
# auto-committed and closed here
```

In [ ]:
# ── DB-API Writing Code — full worked example ──────────────────
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
cur  = conn.cursor()

# ── 1. CREATE ──────────────────────────────────────────────────
cur.execute("""
    CREATE TABLE books (
        id     INTEGER PRIMARY KEY,
        title  TEXT    NOT NULL,
        author TEXT,
        year   INTEGER,
        price  REAL
    )
""")
print("✅ Table created")

# ── 2. executemany — insert multiple rows at once ──────────────
data = [
    (1, "Dune",                    "Frank Herbert",   1965, 12.99),
    (2, "Foundation",              "Isaac Asimov",    1951,  9.99),
    (3, "Neuromancer",             "William Gibson",  1984, 11.50),
    (4, "The Left Hand of Darkness","Ursula K. Le Guin",1969,10.75),
    (5, "Hyperion",                "Dan Simmons",     1989, 13.99),
]
cur.executemany("INSERT INTO books VALUES (?, ?, ?, ?, ?)", data)
conn.commit()
print(f"✅ {cur.rowcount} rows inserted  ← cursor.rowcount")

# ── 3. fetchone ────────────────────────────────────────────────
cur.execute("SELECT * FROM books ORDER BY year")
print("\n── fetchone() ──")
print(cur.fetchone())

# ── 4. fetchmany ──────────────────────────────────────────────
print("\n── fetchmany(2) ──")
print(cur.fetchmany(2))

# ── 5. fetchall ───────────────────────────────────────────────
print("\n── fetchall() (remaining rows) ──")
print(cur.fetchall())

# ── 6. cursor.description → column names ─────────────────────
cur.execute("SELECT * FROM books LIMIT 1")
col_names = [d[0] for d in cur.description]
print("\n── cursor.description → column names ──")
print(col_names)

# ── 7. Parameterized SELECT ───────────────────────────────────
year_filter = 1970
cur.execute("SELECT title, year FROM books WHERE year < ?", (year_filter,))
print(f"\n── Books before {year_filter} (parameterized query) ──")
for row in cur.fetchall():
    print(row)

# ── 8. UPDATE + rowcount ──────────────────────────────────────
cur.execute("UPDATE books SET price = price * 0.9 WHERE year < 1970")
conn.commit()
print(f"\n── 10% discount applied to {cur.rowcount} classic books ──")

# ── 9. Into a DataFrame ───────────────────────────────────────
print("\n── Final table via pd.read_sql_query ──")
df = pd.read_sql_query("SELECT * FROM books ORDER BY year", conn)
display(df)

conn.close()
print("\n✅ Connection closed")